# FILOSOFI Data Transformation: Bronze → Silver

Transform all Filosofi datasets (2014-2021) from Bronze to Silver layer.
- Merge all years into a single standardized dataset
- Rename columns to snake_case
- Handle suppressed data (marked as 's')
- Convert data types appropriately
- Add metadata (department code, commune name, year)

In [70]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

## 1. Load all Filosofi Bronze Files (2014-2021)

In [71]:
# Identify all filosofi bronze files
bronze_path = Path("../../../data/bronze")
filosofi_files = sorted([
    str(bronze_path / f) for f in os.listdir(bronze_path)
    if f.startswith('filosofi_') and f.endswith('_bronze.csv')
])

print(f"Found {len(filosofi_files)} Filosofi Bronze files:")
for f in filosofi_files:
    print(f"  - {Path(f).name}")

Found 8 Filosofi Bronze files:
  - filosofi_2014_rhone_69_bronze.csv
  - filosofi_2015_rhone_69_bronze.csv
  - filosofi_2016_rhone_69_bronze.csv
  - filosofi_2017_rhone_69_bronze.csv
  - filosofi_2018_rhone_69_bronze.csv
  - filosofi_2019_rhone_69_bronze.csv
  - filosofi_2020_rhone_69_bronze.csv
  - filosofi_2021_rhone_69_bronze.csv


In [72]:
# Load each year's data
# Store as dictionary with year as key
dfs_by_year = {}

for file_path in filosofi_files:
    # Extract year from filename (e.g., 'filosofi_2014_rhone_69_bronze.csv' -> 2014)
    filename = Path(file_path).stem
    year = int(filename.split('_')[1])
    
    df = pd.read_csv(file_path, sep=",", dtype=str, nrows=None)
    dfs_by_year[year] = df
    
    print(f"✅ {filename}: shape={df.shape}, year={year}")

print(f"\n📊 Years loaded: {sorted(dfs_by_year.keys())}")

✅ filosofi_2014_rhone_69_bronze: shape=(266, 32), year=2014
✅ filosofi_2015_rhone_69_bronze: shape=(266, 33), year=2015
✅ filosofi_2016_rhone_69_bronze: shape=(265, 33), year=2016
✅ filosofi_2017_rhone_69_bronze: shape=(265, 32), year=2017
✅ filosofi_2018_rhone_69_bronze: shape=(265, 32), year=2018
✅ filosofi_2019_rhone_69_bronze: shape=(264, 32), year=2019
✅ filosofi_2020_rhone_69_bronze: shape=(265, 32), year=2020
✅ filosofi_2021_rhone_69_bronze: shape=(265, 32), year=2021

📊 Years loaded: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


## 2. Standardize Column Names Across Years

In [73]:
# Analyze column patterns across years
# Columns with year suffixes need to be normalized

year_to_suffix = {2014: '14', 2015: '15', 2016: '16', 2017: '17', 2018: '18', 2019: '19', 2020: '20', 2021: '21'}

# Base columns that don't have year suffix
base_cols_2014 = set(dfs_by_year[2014].columns)
print(f"All columns in 2014: {list(base_cols_2014)}")
print(f"\nBase structure inspection:")
print(f"Does 2014 have 'LIBGEO'? {'LIBGEO' in base_cols_2014}")
print(f"Does 2014 have metadata columns? {any(col in base_cols_2014 for col in ['year', 'extraction_source_url', 'ingestion_timestamp', 'source_file_name'])}")

All columns in 2014: ['MED14', 'year', 'NBPERSMENFISC14', 'PIMP14', 'TP60AGE414', 'PTSAC14', 'PPAT14', 'D914', 'PRA14', 'TP60AGE314', 'RD14', 'PPLOGT14', 'TP6014', 'PPMINI14', 'PPFAM14', 'TP60TOL114', 'TP60AGE514', 'PPEN14', 'source_file_name', 'NBMENFISC14', 'PBEN14', 'D114', 'TP60AGE214', 'LIBGEO', 'PIMPOT14', 'TP60AGE114', 'extraction_source_url', 'CODGEO', 'ingestion_timestamp', 'PPSOC14', 'TP60AGE614', 'TP60TOL214']

Base structure inspection:
Does 2014 have 'LIBGEO'? True
Does 2014 have metadata columns? True


In [74]:
def standardize_filosofi_year(df, year):
    """
    Standardize a single year's Filosofi DataFrame.
    - Remove year suffix from column names
    - Add missing LIBGEO if not present (will be handled later via merge)
    - Preserve metadata columns
    """
    df = df.copy()
    suffix = year_to_suffix[year]
    
    # Rename columns by removing year suffix
    rename_map = {}
    for col in df.columns:
        if col.endswith(suffix) and col not in ['year']:
            # Remove suffix from column (e.g., 'NBMENFISC14' -> 'NBMENFISC')
            new_col = col[:-len(suffix)]
            rename_map[col] = new_col
    
    df = df.rename(columns=rename_map)
    
    # Ensure key columns are present
    if 'CODGEO' not in df.columns:
        raise ValueError(f"Year {year}: Missing CODGEO column")
    
    return df

# Apply standardization
dfs_standardized = {}
for year, df in dfs_by_year.items():
    dfs_standardized[year] = standardize_filosofi_year(df, year)
    print(f"✅ Year {year}: Standardized columns")
    print(f"   Sample columns: {dfs_standardized[year].columns[:8].tolist()}")

✅ Year 2014: Standardized columns
   Sample columns: ['CODGEO', 'LIBGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1']
✅ Year 2015: Standardized columns
   Sample columns: ['CODGEO', 'LIBGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1']
✅ Year 2016: Standardized columns
   Sample columns: ['CODGEO', 'LIBGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1']
✅ Year 2017: Standardized columns
   Sample columns: ['CODGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1', 'TP60AGE2']
✅ Year 2018: Standardized columns
   Sample columns: ['CODGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1', 'TP60AGE2']
✅ Year 2019: Standardized columns
   Sample columns: ['CODGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1', 'TP60AGE2']
✅ Year 2020: Standardized columns
   Sample columns: ['CODGEO', 'NBMENFISC', 'NBPERSMENFISC', 'MED', 'PIMP', 'TP60', 'TP60AGE1', 'TP60AGE2']
✅ Year 2021: Standa

## 3. Merge All Years into Single DataFrame

In [75]:
# Create a mapping of CODGEO to LIBGEO from 2014 (which has LIBGEO)
df_2014_with_names = dfs_by_year[2014][['CODGEO', 'LIBGEO']].copy()
df_2014_with_names = df_2014_with_names.drop_duplicates(subset=['CODGEO'])

print(f"Created LIBGEO mapping with {len(df_2014_with_names)} unique communes")
print(f"Sample:")
print(df_2014_with_names.head())

Created LIBGEO mapping with 266 unique communes
Sample:
  CODGEO             LIBGEO
0  69001             Affoux
1  69002         Aigueperse
2  69003  Albigny-sur-Saône
3  69004               Alix
4  69005          Ambérieux


In [76]:
# For each year, add missing LIBGEO via mapping, then collect all
df_silver_list = []

for year in sorted(dfs_standardized.keys()):
    df = dfs_standardized[year].copy()
    
    # Add LIBGEO from mapping if not present
    if 'LIBGEO' not in df.columns:
        df = pd.merge(df, df_2014_with_names, on='CODGEO', how='left')
    
    df_silver_list.append(df)
    print(f"✅ Year {year}: shape={df.shape}")

# Concatenate all years
df_silver = pd.concat(df_silver_list, ignore_index=True)
print(f"\n📊 Combined DataFrame: shape={df_silver.shape}")

✅ Year 2014: shape=(266, 32)
✅ Year 2015: shape=(266, 33)
✅ Year 2016: shape=(265, 33)
✅ Year 2017: shape=(265, 33)
✅ Year 2018: shape=(265, 33)
✅ Year 2019: shape=(264, 33)
✅ Year 2020: shape=(265, 33)
✅ Year 2021: shape=(265, 33)

📊 Combined DataFrame: shape=(2121, 35)


## 4. Data Cleaning & Type Conversion

In [77]:
# Strip whitespace from all string columns
for col in df_silver.columns:
    if df_silver[col].dtype == 'object':
        df_silver[col] = df_silver[col].astype(str).str.strip()

print("✅ Whitespace trimmed from string columns")

✅ Whitespace trimmed from string columns


In [78]:
# Handle suppressed data marked as 's' or 'S'
# Replace with NaN for proper numeric handling
def clean_suppressed_values(val):
    """Replace suppressed data markers with NaN"""
    if isinstance(val, str):
        if val.lower() in ['s', 'np', 'nd']:
            return np.nan
    return val

# Apply to all data columns (skip CODGEO, LIBGEO, year, metadata)
skip_cols = {'CODGEO', 'LIBGEO', 'year', 'extraction_source_url', 'ingestion_timestamp', 'source_file_name'}
numeric_cols_to_clean = [col for col in df_silver.columns if col not in skip_cols]

for col in numeric_cols_to_clean:
    df_silver[col] = df_silver[col].apply(clean_suppressed_values)

print(f"✅ Suppressed data ('s', 'np', 'nd') replaced with NaN")

✅ Suppressed data ('s', 'np', 'nd') replaced with NaN


In [79]:
# Convert CODGEO to string with proper padding
df_silver['CODGEO'] = df_silver['CODGEO'].astype(str).str.strip().str.zfill(5)

# Extract department and commune codes from CODGEO
df_silver['code_departement'] = df_silver['CODGEO'].str[:2]
df_silver['code_commune'] = df_silver['CODGEO'].str[2:5]

# Clean commune name
df_silver['LIBGEO'] = df_silver['LIBGEO'].astype(str).str.strip()

# Ensure year is numeric
df_silver['year'] = pd.to_numeric(df_silver['year'], errors='coerce').astype('Int64')

print("✅ CODGEO extracted to department and commune codes")
print(f"Sample: {df_silver[['CODGEO', 'code_departement', 'code_commune', 'LIBGEO']].head()}")

✅ CODGEO extracted to department and commune codes
Sample:   CODGEO code_departement code_commune             LIBGEO
0  69001               69          001             Affoux
1  69002               69          002         Aigueperse
2  69003               69          003  Albigny-sur-Saône
3  69004               69          004               Alix
4  69005               69          005          Ambérieux


In [80]:
# Convert all data columns to numeric
numeric_cols = [col for col in df_silver.columns if col not in skip_cols]

for col in numeric_cols:
    # Handle European decimal format (comma separator)
    df_silver[col] = df_silver[col].astype(str).str.replace(',', '.', regex=False)
    df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

print(f"✅ Converted {len(numeric_cols)} numeric columns")
print(f"Data types sample: {df_silver[['NBMENFISC', 'NBPERSMENFISC', 'MED']].dtypes.to_dict()}")

✅ Converted 31 numeric columns
Data types sample: {'NBMENFISC': dtype('float64'), 'NBPERSMENFISC': dtype('float64'), 'MED': dtype('float64')}


## 5. Rename Columns to Snake_Case

In [81]:
# Create comprehensive rename mapping
rename_map_final = {
    'CODGEO': 'code_insee_commune',
    'LIBGEO': 'libelle_commune',
    'NBMENFISC': 'nb_menages_fiscaux',
    'NBPERSMENFISC': 'nb_personnes_menages_fiscaux',
    'MED': 'mediane_niveau_vie_euros',
    'PIMP': 'pct_menages_imposables',
    'TP60': 'taux_pauvrete_ensemble',
    'TP60AGE1': 'taux_pauvrete_moins_30ans',
    'TP60AGE2': 'taux_pauvrete_30_39ans',
    'TP60AGE3': 'taux_pauvrete_40_49ans',
    'TP60AGE4': 'taux_pauvrete_50_59ans',
    'TP60AGE5': 'taux_pauvrete_60_74ans',
    'TP60AGE6': 'taux_pauvrete_75ans_plus',
    'TP60TOL1': 'taux_pauvrete_proprietaires',
    'TP60TOL2': 'taux_pauvrete_locataires',
    'PRA': 'pct_revenus_activite',
    'PTSA': 'pct_revenus_salaires_chomage',
    'PACT': 'pct_revenus_activite_non_salariee',  # Used in 2015+
    'PCHO': 'pct_revenus_activite_non_salariee',  # Alternative name in 2015+
    'PBEN': 'pct_revenus_non_salaries',
    'PPEN': 'pct_pensions_retraites',
    'PPAT': 'pct_revenus_patrimoine',
    'PPSOC': 'pct_prestations_sociales',
    'PPFAM': 'pct_prestations_familiales',
    'PPMINI': 'pct_minima_sociaux',
    'PPLOGT': 'pct_prestations_logement',
    'PIMPOT': 'pct_impots',
    'D1': 'decile_1_revenu',
    'D9': 'decile_9_revenu',
    'RD': 'ratio_deciles',
    'year': 'annee',
    'extraction_source_url': 'source_extraction_url',
    'ingestion_timestamp': 'date_ingestion',
    'source_file_name': 'fichier_source',
}

print(f"Rename mapping created with {len(rename_map_final)} entries")

Rename mapping created with 34 entries


In [82]:
# Apply rename, but only for columns that exist
actual_rename_map = {k: v for k, v in rename_map_final.items() if k in df_silver.columns}
df_silver = df_silver.rename(columns=actual_rename_map)

# Handle PTSAC separately if it wasn't renamed
if 'PTSAC' in df_silver.columns:
    df_silver = df_silver.rename(columns={'PTSAC': 'pct_revenus_salaires_chomage'})

print(f"✅ Renamed {len(actual_rename_map)} columns")
print(f"\nNew columns:")
print(df_silver.columns.tolist())

✅ Renamed 34 columns

New columns:
['code_insee_commune', 'libelle_commune', 'nb_menages_fiscaux', 'nb_personnes_menages_fiscaux', 'mediane_niveau_vie_euros', 'pct_menages_imposables', 'taux_pauvrete_ensemble', 'taux_pauvrete_moins_30ans', 'taux_pauvrete_30_39ans', 'taux_pauvrete_40_49ans', 'taux_pauvrete_50_59ans', 'taux_pauvrete_60_74ans', 'taux_pauvrete_75ans_plus', 'taux_pauvrete_proprietaires', 'taux_pauvrete_locataires', 'pct_revenus_activite', 'pct_revenus_salaires_chomage', 'pct_revenus_non_salaries', 'pct_pensions_retraites', 'pct_revenus_patrimoine', 'pct_prestations_sociales', 'pct_prestations_familiales', 'pct_minima_sociaux', 'pct_prestations_logement', 'pct_impots', 'decile_1_revenu', 'decile_9_revenu', 'ratio_deciles', 'annee', 'source_extraction_url', 'date_ingestion', 'fichier_source', 'pct_revenus_activite_non_salariee', 'pct_revenus_salaires_chomage', 'pct_revenus_activite_non_salariee', 'code_departement', 'code_commune']


In [83]:
# Some years have both PACT/PCHO (same meaning)
# These are non-salary business income  
# We keep only one consolidated column
pact_cols = [col for col in df_silver.columns if 'PACT' in col or 'PCHO' in col]
if len(pact_cols) > 0:
    # Create consolidated column: pct_revenus_activite_non_salariee
    if 'pct_revenus_activite_non_salariee' not in df_silver.columns:
        df_silver['pct_revenus_activite_non_salariee'] = np.nan
    
    # Fill with non-null values from either PACT or PCHO columns
    for col in pact_cols:
        mask = df_silver['pct_revenus_activite_non_salariee'].isna()
        df_silver.loc[mask, 'pct_revenus_activite_non_salariee'] = df_silver.loc[mask, col]
    
    # Remove the original columns
    cols_to_drop = [col for col in pact_cols if col != 'pct_revenus_activite_non_salariee']
    if cols_to_drop:
        df_silver = df_silver.drop(columns=cols_to_drop)
    
print("✅ Handled duplicate column mappings (PACT/PCHO consolidated)")

✅ Handled duplicate column mappings (PACT/PCHO consolidated)


## 6. Add Metadata Columns

In [84]:
# Add source and refresh date
df_silver['source_dataset'] = 'filosofi_commune'
df_silver['date_refresh'] = pd.to_datetime(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

print("✅ Added metadata columns")

✅ Added metadata columns


In [85]:
# Remove duplicate columns (keep first occurrence)
before_dedup = len(df_silver.columns)
df_silver = df_silver.loc[:, ~df_silver.columns.duplicated(keep='first')]
after_dedup = len(df_silver.columns)

print(f"✅ Removed duplicate columns: {before_dedup} → {after_dedup}")
print(f"Remaining columns: {df_silver.columns.tolist()}")

✅ Removed duplicate columns: 39 → 37
Remaining columns: ['code_insee_commune', 'libelle_commune', 'nb_menages_fiscaux', 'nb_personnes_menages_fiscaux', 'mediane_niveau_vie_euros', 'pct_menages_imposables', 'taux_pauvrete_ensemble', 'taux_pauvrete_moins_30ans', 'taux_pauvrete_30_39ans', 'taux_pauvrete_40_49ans', 'taux_pauvrete_50_59ans', 'taux_pauvrete_60_74ans', 'taux_pauvrete_75ans_plus', 'taux_pauvrete_proprietaires', 'taux_pauvrete_locataires', 'pct_revenus_activite', 'pct_revenus_salaires_chomage', 'pct_revenus_non_salaries', 'pct_pensions_retraites', 'pct_revenus_patrimoine', 'pct_prestations_sociales', 'pct_prestations_familiales', 'pct_minima_sociaux', 'pct_prestations_logement', 'pct_impots', 'decile_1_revenu', 'decile_9_revenu', 'ratio_deciles', 'annee', 'source_extraction_url', 'date_ingestion', 'fichier_source', 'pct_revenus_activite_non_salariee', 'code_departement', 'code_commune', 'source_dataset', 'date_refresh']


## 7. Organize Final Column Order

In [86]:
# Define column order: geography, time, aggregate measures, breakdown by age, by tenure, by income source
final_column_order = [
    # Geography
    'code_departement',
    'code_commune',
    'code_insee_commune',
    'libelle_commune',
    
    # Time
    'annee',
    
    # Basic household metrics
    'nb_menages_fiscaux',
    'nb_personnes_menages_fiscaux',
    'mediane_niveau_vie_euros',
    'pct_menages_imposables',
    
    # Poverty rates
    'taux_pauvrete_ensemble',
    'taux_pauvrete_moins_30ans',
    'taux_pauvrete_30_39ans',
    'taux_pauvrete_40_49ans',
    'taux_pauvrete_50_59ans',
    'taux_pauvrete_60_74ans',
    'taux_pauvrete_75ans_plus',
    'taux_pauvrete_proprietaires',
    'taux_pauvrete_locataires',
    
    # Income composition
    'pct_revenus_activite',
    'pct_revenus_salaires_chomage',
    'pct_revenus_activite_non_salariee',
    'pct_pensions_retraites',
    'pct_revenus_patrimoine',
    'pct_prestations_sociales',
    'pct_prestations_familiales',
    'pct_minima_sociaux',
    'pct_prestations_logement',
    'pct_impots',
    
    # Deciles
    'decile_1_revenu',
    'decile_9_revenu',
    'ratio_deciles',
    
    # Non-salary income
    'pct_revenus_non_salaries',
    
    # Metadata
    'source_dataset',
    'date_refresh',
    'source_extraction_url',
    'date_ingestion',
    'fichier_source',
]

# Filter to only columns that exist
final_column_order = [col for col in final_column_order if col in df_silver.columns]

# Add any remaining columns not in the order
remaining_cols = [col for col in df_silver.columns if col not in final_column_order]
if remaining_cols:
    print(f"⚠️ Columns not in order specification: {remaining_cols}")
    print("Adding them to the end...")
    final_column_order.extend(remaining_cols)

df_silver = df_silver[final_column_order]

print(f"✅ Reorganized columns: {len(final_column_order)} columns")
print(f"\nFinal column order (first 10):")
for i, col in enumerate(final_column_order[:10], 1):
    print(f"  {i}. {col}")

✅ Reorganized columns: 37 columns

Final column order (first 10):
  1. code_departement
  2. code_commune
  3. code_insee_commune
  4. libelle_commune
  5. annee
  6. nb_menages_fiscaux
  7. nb_personnes_menages_fiscaux
  8. mediane_niveau_vie_euros
  9. pct_menages_imposables
  10. taux_pauvrete_ensemble


## 8. Data Quality Checks

In [87]:
print("📊 DATA QUALITY REPORT")
print("="*60)
print(f"\nTotal rows: {len(df_silver):,}")
print(f"Total columns: {len(df_silver.columns)}")
print(f"\nUnique communes: {df_silver['code_insee_commune'].nunique()}")
print(f"Unique years: {sorted(df_silver['annee'].dropna().unique().astype(int).tolist())}")
print(f"Year range: {df_silver['annee'].min():.0f} - {df_silver['annee'].max():.0f}")
print(f"\nRecords per year:")
for year in sorted(df_silver['annee'].dropna().unique()):
    count = len(df_silver[df_silver['annee'] == year])
    print(f"  {int(year)}: {count:,} records")

print(f"\nMissing values (top 10):")
missing = df_silver.isnull().sum()
missing_sorted = missing.sort_values(ascending=False).head(10)
for col, count in missing_sorted.items():
    pct = (count / len(df_silver)) * 100
    print(f"  {col}: {count:,} ({pct:.1f}%)")

print(f"\nData type distribution:")
for dtype in df_silver.dtypes.unique():
    count = (df_silver.dtypes == dtype).sum()
    print(f"  {dtype}: {count} columns")

📊 DATA QUALITY REPORT

Total rows: 2,121
Total columns: 37

Unique communes: 266
Unique years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Year range: 2014 - 2021

Records per year:
  2014: 266 records
  2015: 266 records
  2016: 265 records
  2017: 265 records
  2018: 265 records
  2019: 264 records
  2020: 265 records
  2021: 265 records

Missing values (top 10):
  taux_pauvrete_75ans_plus: 2,023 (95.4%)
  pct_revenus_salaires_chomage: 2,002 (94.4%)
  pct_revenus_activite: 2,002 (94.4%)
  taux_pauvrete_moins_30ans: 2,000 (94.3%)
  taux_pauvrete_60_74ans: 1,983 (93.5%)
  taux_pauvrete_50_59ans: 1,919 (90.5%)
  taux_pauvrete_proprietaires: 1,886 (88.9%)
  taux_pauvrete_30_39ans: 1,875 (88.4%)
  taux_pauvrete_40_49ans: 1,838 (86.7%)
  taux_pauvrete_locataires: 1,676 (79.0%)

Data type distribution:
  int64: 2 columns
  object: 6 columns
  Int64: 1 columns
  float64: 27 columns
  datetime64[ns]: 1 columns


In [88]:
print("\n🔍 VALIDATION CHECKS")
print("="*60)

# Check for duplicates
duplicates = df_silver.duplicated(subset=['code_insee_commune', 'annee']).sum()
print(f"\nDuplicate (commune, year) pairs: {duplicates}")
if duplicates > 0:
    print("⚠️ WARNING: Found duplicate records!")
    print(df_silver[df_silver.duplicated(subset=['code_insee_commune', 'annee'], keep=False)].head())
else:
    print("✅ No duplicates found")

# Check for valid ranges
print(f"\n✅ Year values valid: {df_silver['annee'].min() >= 2014 and df_silver['annee'].max() <= 2021}")
print(f"✅ Department code is '69': {df_silver['code_departement'].unique().tolist()}")

# Check for poverty rates in valid range [0, 100]
poverty_cols = [col for col in df_silver.columns if 'taux_pauvrete' in col]
if poverty_cols:
    for col in poverty_cols[:3]:  # Check first 3
        valid = ((df_silver[col] >= 0) | df_silver[col].isna()).all()
        print(f"✅ {col} values valid: {valid}")


🔍 VALIDATION CHECKS

Duplicate (commune, year) pairs: 0
✅ No duplicates found

✅ Year values valid: True
✅ Department code is '69': [69]
✅ taux_pauvrete_ensemble values valid: True
✅ taux_pauvrete_moins_30ans values valid: True
✅ taux_pauvrete_30_39ans values valid: True


In [89]:
# Show sample data
print("\n📋 SAMPLE DATA (2014, 2017, 2021)")
print("="*60)

for year in [2014, 2017, 2021]:
    sample = df_silver[df_silver['annee'] == year].iloc[0:2]
    print(f"\n{int(year)}:")
    display(sample[['code_insee_commune', 'libelle_commune', 'annee', 'nb_menages_fiscaux', 'mediane_niveau_vie_euros', 'taux_pauvrete_ensemble']])


📋 SAMPLE DATA (2014, 2017, 2021)

2014:


,code_insee_commune,libelle_commune,annee,nb_menages_fiscaux,mediane_niveau_vie_euros,taux_pauvrete_ensemble
0,69001,Affoux,2014,124.0,21462.857143,NaN
1,69002,Aigueperse,2014,113.0,18494.500000,NaN



2017:


,code_insee_commune,libelle_commune,annee,nb_menages_fiscaux,mediane_niveau_vie_euros,taux_pauvrete_ensemble
797,69001,Affoux,2017,136.0,22610.0,NaN
798,69002,Aigueperse,2017,115.0,19550.0,NaN



2021:


,code_insee_commune,libelle_commune,annee,nb_menages_fiscaux,mediane_niveau_vie_euros,taux_pauvrete_ensemble
1856,69001,Affoux,2021,138.0,24970.0,NaN
1857,69002,Aigueperse,2021,112.0,21290.0,NaN


## 9. Save Silver Dataset

In [90]:
# Define output path
silver_path = Path("../../../data/silver/filosofi_2014_2021_commune_silver.csv")

# Create parent directory if needed
silver_path.parent.mkdir(parents=True, exist_ok=True)

# Save to CSV
df_silver.to_csv(silver_path, index=False, sep=";", encoding="utf-8")

print(f"✅ Dataset saved successfully!")
print(f"\n📁 Output file: {silver_path}")
print(f"📊 Shape: {df_silver.shape}")
print(f"💾 File size: {silver_path.stat().st_size / 1024:.1f} KB")

✅ Dataset saved successfully!

📁 Output file: ../../../data/silver/filosofi_2014_2021_commune_silver.csv
📊 Shape: (2121, 37)
💾 File size: 599.4 KB


In [91]:
print("\n" + "="*60)
print("✅ TRANSFORMATION COMPLETE")
print("="*60)
print(f"\n📈 Summary:")
print(f"  Input:  8 bronze files (2014-2021)")
print(f"  Output: 1 silver file (filosofi_2014_2021_commune_silver.csv)")
print(f"  Records: {len(df_silver):,} (communes × years)")
print(f"  Columns: {len(df_silver.columns)} (all standardized)")
print(f"  Geography: Rhône (69) department")
print(f"  Time span: 2014-2021")
print(f"\n🎯 Data is ready for Gold layer aggregation and feature engineering.")


✅ TRANSFORMATION COMPLETE

📈 Summary:
  Input:  8 bronze files (2014-2021)
  Output: 1 silver file (filosofi_2014_2021_commune_silver.csv)
  Records: 2,121 (communes × years)
  Columns: 37 (all standardized)
  Geography: Rhône (69) department
  Time span: 2014-2021

🎯 Data is ready for Gold layer aggregation and feature engineering.
